In [1]:
# Install the small set of libraries we need
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate

import os
import re
import time
import math
import requests
import textwrap
import numpy as np
import torch
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Track total runtime
overall_start = time.time()

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 24.3 MB/s eta 0:00:00
Device: cuda


In [2]:
# Download a real public PDF automatically
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"  # Attention Is All You Need
pdf_path = "/content/attention_is_all_you_need.pdf"

headers = {"User-Agent": "Mozilla/5.0"}
r = requests.get(pdf_url, headers=headers, timeout=60)
r.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(r.content)

print("Downloaded PDF:", pdf_path, "| Size:", round(os.path.getsize(pdf_path) / 1024, 2), "KB")

# Extract text from the PDF
reader = PdfReader(pdf_path)
raw_pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    text = re.sub(r"\s+", " ", text).strip()
    if text:
        raw_pages.append(text)

raw_text = "\n".join(raw_pages)

print("Pages in PDF:", len(reader.pages))
print("Pages with extracted text:", len(raw_pages))
print("Raw text length:", len(raw_text))

print("\n--- Text preview ---")
print(textwrap.shorten(raw_text[:1200], width=1200, placeholder=" ..."))

Downloaded PDF: /content/attention_is_all_you_need.pdf | Size: 2163.32 KB
Pages in PDF: 15
Pages with extracted text: 15
Raw text length: 39608

--- Text preview ---
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mec

In [3]:
# Simple fixed-size chunking with overlap
# Chunk size 1000 and overlap 150 keeps meaning intact while staying fast for retrieval and generation
def make_chunks(text, chunk_size=1000, overlap=150):
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if len(chunk) > 100:
            chunks.append(chunk)
    return chunks

chunk_size = 1000
overlap = 150
chunks = make_chunks(raw_text, chunk_size=chunk_size, overlap=overlap)

print("Chunk size:", chunk_size)
print("Overlap:", overlap)
print("Number of chunks:", len(chunks))

print("\n--- First 2 chunk previews ---")
for i in range(min(2, len(chunks))):
    print(f"\nChunk {i+1}:")
    print(textwrap.shorten(chunks[i], width=500, placeholder=" ..."))

Chunk size: 1000
Overlap: 150
Number of chunks: 47

--- First 2 chunk previews ---

Chunk 1:
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu ...

Chunk 2:
se a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, im

In [4]:
# Load a lightweight embedding model
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name, device=device)

embedding_start = time.time()
chunk_embeddings = embedder.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
embedding_time = time.time() - embedding_start

print("Embedding model:", embed_model_name)
print("Embedding shape:", chunk_embeddings.shape)
print("Embedding vector dimension:", chunk_embeddings.shape[1])
print("Embedding time (sec):", round(embedding_time, 2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding shape: (47, 384)
Embedding vector dimension: 384
Embedding time (sec): 1.22


In [5]:
# Build a simple FAISS index for fast similarity search
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # cosine similarity works because embeddings are normalized
index.add(chunk_embeddings.astype(np.float32))

print("Vector store: FAISS IndexFlatIP")
print("Vectors stored:", index.ntotal)

Vector store: FAISS IndexFlatIP
Vectors stored: 47


In [6]:
def retrieve_vector(query, top_k=3):
    query_embedding = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        results.append({
            "chunk_id": int(idx),
            "score": float(score),
            "text": chunks[int(idx)]
        })
    return results

def keyword_overlap_score(query, chunk_text):
    q_words = set(re.findall(r"\b[a-zA-Z]{3,}\b", query.lower()))
    c_words = set(re.findall(r"\b[a-zA-Z]{3,}\b", chunk_text.lower()))
    if not q_words:
        return 0.0
    return len(q_words & c_words) / len(q_words)

def retrieve_hybrid(query, top_k=3, candidate_k=6):
    # First get vector candidates, then re-rank with a tiny keyword overlap signal
    candidates = retrieve_vector(query, top_k=candidate_k)
    reranked = []
    for item in candidates:
        kw = keyword_overlap_score(query, item["text"])
        combined = 0.8 * item["score"] + 0.2 * kw
        reranked.append({**item, "keyword_score": kw, "combined_score": combined})
    reranked.sort(key=lambda x: x["combined_score"], reverse=True)
    return reranked[:top_k]

test_query = "Why is positional encoding needed in the Transformer?"
print("\n--- Retrieval sanity check ---")
for item in retrieve_hybrid(test_query, top_k=3):
    print(f"\nChunk ID: {item['chunk_id']} | vector={item['score']:.4f} | keyword={item['keyword_score']:.4f} | combined={item['combined_score']:.4f}")
    print(textwrap.shorten(item["text"], width=420, placeholder=" ..."))


--- Retrieval sanity check ---

Chunk ID: 8 | vector=0.4780 | keyword=0.3333 | combined=0.4491
, ..., xn) to a sequence of continuous representations z = (z1, ..., zn). Given z, the decoder then generates an output sequence (y1, ..., ym) of symbols one element at a time. At each step the model is auto-regressive [10], consuming the previously generated symbols as additional input when generating the next. 2 Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture ...

Chunk ID: 29 | vector=0.3939 | keyword=0.6667 | combined=0.4485
tions on the Transformer architecture. Unlisted values are identical to those of the base model. All metrics are on the English-to-German translation development set, newstest2013. Listed perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to per-word perplexities. N d model dff h d k dv Pdrop ϵls train PPL BLEU params steps (dev) (dev) ×106 base 6 512 2048 8 64 64 0.1 ...



In [7]:
# Use a small text-to-text model so the notebook runs quickly on free Colab T4
gen_model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name).to(device)

print("Generation model:", gen_model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generation model: google/flan-t5-small


In [8]:
def build_prompt(question, retrieved_chunks):
    context = "\n\n".join([f"Chunk {i+1}: {c['text']}" for i, c in enumerate(retrieved_chunks)])
    prompt = f"""
You are a helpful assistant.
Answer the question only using the context below.
If the answer is not in the context, say you could not find it in the document.

Context:
{context}

Question: {question}
Answer:
"""
    return prompt.strip()

def generate_answer(question, top_k=3):
    retrieval_start = time.time()
    retrieved = retrieve_hybrid(question, top_k=top_k)
    retrieval_time = time.time() - retrieval_start

    prompt = build_prompt(question, retrieved)

    gen_start = time.time()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        num_beams=2
    )
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generation_time = time.time() - gen_start

    return {
        "answer": answer,
        "retrieved": retrieved,
        "retrieval_time": retrieval_time,
        "generation_time": generation_time
    }

sample_questions = [
    "Why is self-attention important in the Transformer model?",
    "What problem does positional encoding solve?",
    "What does the paper say about training or optimization?"
]

qa_timings = []

for q in sample_questions:
    print("\n" + "="*100)
    print("Question:", q)
    result = generate_answer(q, top_k=3)
    qa_timings.append((result["retrieval_time"], result["generation_time"]))

    print(f"Retrieval time: {result['retrieval_time']:.3f} sec")
    print(f"Generation time: {result['generation_time']:.3f} sec")

    print("\nRetrieved chunks:")
    for item in result["retrieved"]:
        print(f"\n- Chunk ID {item['chunk_id']} | combined={item['combined_score']:.4f}")
        print(textwrap.shorten(item["text"], width=380, placeholder=" ..."))

    print("\nGrounded answer:")
    print(result["answer"])


Question: Why is self-attention important in the Transformer model?
Retrieval time: 0.072 sec
Generation time: 4.592 sec

Retrieved chunks:

- Chunk ID 14 | combined=0.6204
esW Q i ∈ Rdmodel×dk, W K i ∈ Rdmodel×dk, W V i ∈ Rdmodel×dv and W O ∈ Rhdv×dmodel. In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in ...

- Chunk ID 8 | combined=0.5610
, ..., xn) to a sequence of continuous representations z = (z1, ..., zn). Given z, the decoder then generates an output sequence (y1, ..., ym) of symbols one element at a time. At each step the model is auto-regressive [10], consuming the previously generated symbols as additional input when generating the next. 2 Figure 1: The Transformer - model architecture. The ...

- Chunk ID 2 | combined=0.5397
oth with 

In [9]:
# Compare pure vector retrieval vs the simple hybrid re-rank for one question
experiment_question = "How does the model use attention to handle long-range dependencies?"

vector_only = retrieve_vector(experiment_question, top_k=3)
hybrid = retrieve_hybrid(experiment_question, top_k=3, candidate_k=6)

print("Question:", experiment_question)

print("\n--- Vector only top chunks ---")
for item in vector_only:
    print(f"\nChunk ID {item['chunk_id']} | score={item['score']:.4f}")
    print(textwrap.shorten(item["text"], width=360, placeholder=" ..."))

print("\n--- Hybrid re-ranked top chunks ---")
for item in hybrid:
    print(f"\nChunk ID {item['chunk_id']} | vector={item['score']:.4f} | keyword={item['keyword_score']:.4f} | combined={item['combined_score']:.4f}")
    print(textwrap.shorten(item["text"], width=360, placeholder=" ..."))

print("\nObservation: hybrid retrieval is a tiny improvement step because it keeps the fast vector search, then nudges results with keyword overlap.")

Question: How does the model use attention to handle long-range dependencies?

--- Vector only top chunks ---

Chunk ID 20 | score=0.5572
r three desiderata. One is the total computational complexity per layer. Another is the amount of computation that can be parallelized, as measured by the minimum number of sequential operations required. The third is the path length between long-range dependencies in the network. Learning long-range dependencies is a key challenge in many sequence ...

Chunk ID 4 | score=0.5523
ligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across ...

Chunk ID 22 | score=0.5153
A single convolutional layer with kernel width k < ndoes not connect all pairs of input and

In [10]:
retrieval_times = [x[0] for x in qa_timings]
generation_times = [x[1] for x in qa_timings]

metrics = {
    "chunk_size": chunk_size,
    "overlap": overlap,
    "num_chunks": len(chunks),
    "embedding_model": embed_model_name,
    "embedding_dim": dimension,
    "vector_store": "FAISS IndexFlatIP",
    "generation_model": gen_model_name,
    "embedding_time_sec": embedding_time,
    "avg_retrieval_time_sec": float(np.mean(retrieval_times)),
    "avg_generation_time_sec": float(np.mean(generation_times)),
    "total_runtime_sec": time.time() - overall_start
}

metrics

{'chunk_size': 1000,
 'overlap': 150,
 'num_chunks': 47,
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'embedding_dim': 384,
 'vector_store': 'FAISS IndexFlatIP',
 'generation_model': 'google/flan-t5-small',
 'embedding_time_sec': 1.2237420082092285,
 'avg_retrieval_time_sec': 0.032570600509643555,
 'avg_generation_time_sec': 1.6516650517781575,
 'total_runtime_sec': 22.392133712768555}

In [11]:
print("SYSTEM METRICS REPORT")
print("="*60)
print(f"Chunking profile      : chunk size = {metrics['chunk_size']}, overlap = {metrics['overlap']}, chunks = {metrics['num_chunks']}")
print(f"Embedding model       : {metrics['embedding_model']}")
print(f"Embedding dimension   : {metrics['embedding_dim']}")
print(f"Vector store          : {metrics['vector_store']}")
print(f"Language model        : {metrics['generation_model']}")
print(f"Embedding time        : {metrics['embedding_time_sec']:.2f} sec")
print(f"Avg retrieval time    : {metrics['avg_retrieval_time_sec']:.3f} sec")
print(f"Avg generation time   : {metrics['avg_generation_time_sec']:.3f} sec")
print(f"Total pipeline runtime: {metrics['total_runtime_sec']:.2f} sec")

SYSTEM METRICS REPORT
Chunking profile      : chunk size = 1000, overlap = 150, chunks = 47
Embedding model       : sentence-transformers/all-MiniLM-L6-v2
Embedding dimension   : 384
Vector store          : FAISS IndexFlatIP
Language model        : google/flan-t5-small
Embedding time        : 1.22 sec
Avg retrieval time    : 0.033 sec
Avg generation time   : 1.652 sec
Total pipeline runtime: 22.39 sec
